<a href="https://colab.research.google.com/github/ViSaReVe/triton-serving-lab/blob/main/Deliverable1_Step2_3_triton_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deliverable 1 · Steps 2 & 3 — Serve on Triton + benchmark (Colab T4)

Your Mac is **arm64**, so the official Triton container would be a ~15 GB x86 image under emulation — slow, and any
benchmark from it would be meaningless. Instead we run **real Triton** here on Colab's x86 + T4 via **PyTriton**
(`pip install nvidia-pytriton` ships the actual Triton server binary — no Docker required).

**Step 2:** stand up Triton, send your first inference request.  
**Step 3:** benchmark — dynamic batching ON vs OFF, and GPU vs CPU — to fill your README table.

Runtime → Change runtime type → **T4 GPU**.


## 0. Setup
`onnxruntime-gpu`'s newest build targets CUDA 13, which Colab may not have (that's the `libcudart.so.13` error from
Step 1). We pin a CUDA-12 build and **fall back to CPU automatically**, then print which provider is actually active —
always verify this rather than assuming you're on GPU.


In [1]:
!nvidia-smi -L
!pip -q install nvidia-pytriton
!pip -q uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null
!pip -q install 'onnxruntime-gpu==1.19.2' || pip -q install onnxruntime
import numpy as np, onnxruntime as ort
print('onnxruntime', ort.__version__)
print('available providers:', ort.get_available_providers())


GPU 0: Tesla T4 (UUID: GPU-867c35f1-77a3-85e7-aac2-23c057269510)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.8/111.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.9/613.9 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.8 M

## 1. Get the model
Upload `audio_cnn.onnx` from Step 1. (Once your repo is pushed you can instead clone it — see the commented line.)


In [2]:
from google.colab import files
up = files.upload()                 # pick audio_cnn.onnx
ONNX = [f for f in up if f.endswith('.onnx')][0]
# alternative once pushed:
# !git clone https://github.com/ViSaReVe/triton-serving-lab.git
# ONNX = 'triton-serving-lab/model_repository/audio_cnn/1/model.onnx'
print('model:', ONNX)


Saving audio_cnn.onnx to audio_cnn.onnx
model: audio_cnn.onnx


## 2. Load the ONNX on the GPU
We create the inference session Triton will call. Note we **check the provider actually in use** — if CUDA didn't
load, you'd silently benchmark CPU and think it was GPU.


In [3]:
providers = ['CUDAExecutionProvider','CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
sess_gpu = ort.InferenceSession(ONNX, providers=providers)
ACTIVE = sess_gpu.get_providers()[0]
print('ACTIVE PROVIDER:', ACTIVE)      # CUDAExecutionProvider = really on the T4

# CPU-only session for the CPU-vs-GPU comparison in Step 3
sess_cpu = ort.InferenceSession(ONNX, providers=['CPUExecutionProvider'])

x = np.random.randn(4,1,128,128).astype('float32')
print('smoke test ->', sess_gpu.run(['logits'], {'input': x})[0].shape)   # (4,10)


ACTIVE PROVIDER: CUDAExecutionProvider


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 3. STEP 2 — bind the model to Triton and start the server
This is real Triton. Two models, identical weights, differing only in the batcher:
- **audio_cnn** — `DynamicBatcher(preferred_batch_size=[4,8,16], max_queue_delay_microseconds=3000)`
- **audio_cnn_nobatch** — `max_batch_size=1`, no fusing (the control group)

Same knobs as your `config.pbtxt`, expressed in Python. `@batch` makes Triton hand your function a *fused batch*.
`triton.run()` is non-blocking so we can send requests from the same notebook.


In [ ]:
from pytriton.decorators import batch
from pytriton.model_config import ModelConfig, Tensor, DynamicBatcher
from pytriton.triton import Triton

@batch
def infer_batched(input):
    return {'logits': sess_gpu.run(['logits'], {'input': input})[0]}

@batch
def infer_nobatch(input):
    return {'logits': sess_gpu.run(['logits'], {'input': input})[0]}

triton = Triton()
triton.bind(model_name='audio_cnn', infer_func=infer_batched,
    inputs=[Tensor(name='input', dtype=np.float32, shape=(1,128,128))],
    outputs=[Tensor(name='logits', dtype=np.float32, shape=(10,))],
    config=ModelConfig(max_batch_size=32,
        batcher=DynamicBatcher(preferred_batch_size=[4,8,16], max_queue_delay_microseconds=3000)))
triton.bind(model_name='audio_cnn_nobatch', infer_func=infer_nobatch,
    inputs=[Tensor(name='input', dtype=np.float32, shape=(1,128,128))],
    outputs=[Tensor(name='logits', dtype=np.float32, shape=(10,))],
    config=ModelConfig(max_batch_size=1))

triton.run()      # non-blocking; HTTP on :8000, gRPC :8001, metrics :8002
import time; time.sleep(5)
print('Triton is up.')


### Your first inference request
The Step-2 milestone: your trained EE541 model answering over HTTP through Triton.


In [ ]:
import tritonclient.http as httpclient
client = httpclient.InferenceServerClient(url='localhost:8000')
print('server live:', client.is_server_live())
for m in ('audio_cnn','audio_cnn_nobatch'):
    print(f'  {m} ready:', client.is_model_ready(m))

x1 = np.random.randn(1,1,128,128).astype('float32')
inp = httpclient.InferInput('input', x1.shape, 'FP32'); inp.set_data_from_numpy(x1)
r = client.infer('audio_cnn', inputs=[inp], outputs=[httpclient.InferRequestedOutput('logits')])
logits = r.as_numpy('logits')
print('logits:', logits.shape, '| predicted class:', int(logits.argmax()))


## 4. STEP 3 — benchmark
Fire many concurrent **single-sample** requests and time each. That's the realistic serving pattern: many independent
clients. Dynamic batching wins by *fusing* those into one forward pass — fewer, larger passes use the GPU far better.


In [ ]:
import time, statistics
from concurrent.futures import ThreadPoolExecutor, as_completed

def pct(v,p):
    s=sorted(v); k=max(0,min(len(s)-1,int(round((p/100)*(len(s)-1))))); return s[k]

def bench(model, n=400, conc=16):
    cl = httpclient.InferenceServerClient(url='localhost:8000', concurrency=conc+8)
    xx = np.random.randn(1,1,128,128).astype('float32')
    def one():
        i = httpclient.InferInput('input', xx.shape, 'FP32'); i.set_data_from_numpy(xx)
        t=time.perf_counter(); cl.infer(model, inputs=[i], outputs=[httpclient.InferRequestedOutput('logits')])
        return (time.perf_counter()-t)*1000
    one()  # warm-up
    t0=time.perf_counter()
    with ThreadPoolExecutor(max_workers=conc) as ex:
        lat=[f.result() for f in as_completed([ex.submit(one) for _ in range(n)])]
    wall=time.perf_counter()-t0
    return {'model':model,'conc':conc,'rps':round(n/wall,1),'p50':round(pct(lat,50),2),
            'p95':round(pct(lat,95),2),'p99':round(pct(lat,99),2)}

rows=[]
for conc in (1,8,16,32):
    for m in ('audio_cnn','audio_cnn_nobatch'):
        r=bench(m, n=400, conc=conc); rows.append(r); print(r)


### Results table (paste into your README)


In [ ]:
print(f'ACTIVE PROVIDER: {ACTIVE}\n')
print('| model | concurrency | throughput (req/s) | p50 ms | p95 ms | p99 ms |')
print('|---|---|---|---|---|---|')
for r in rows:
    name = 'dynamic batching' if r['model']=='audio_cnn' else 'no batching'
    print(f"| {name} | {r['conc']} | {r['rps']} | {r['p50']} | {r['p95']} | {r['p99']} |")


### GPU vs CPU (same model, same ONNX)
Direct engine-level comparison, no serving overhead — isolates the hardware effect.


In [ ]:
def raw_bench(sess, bs=8, iters=100):
    xx=np.random.randn(bs,1,128,128).astype('float32')
    sess.run(['logits'],{'input':xx})
    t0=time.perf_counter()
    for _ in range(iters): sess.run(['logits'],{'input':xx})
    dt=(time.perf_counter()-t0)/iters*1000
    return round(dt,2), round(bs*iters/((dt/1000)*iters),1)

for label, s in (('GPU '+ACTIVE, sess_gpu), ('CPU', sess_cpu)):
    ms, rps = raw_bench(s)
    print(f'{label:32s} batch=8: {ms:7.2f} ms/batch  ~{rps:8.1f} samples/s')


## 5. Shut down + what you learned


In [ ]:
triton.stop()
print('Triton stopped.')


**What Step 2–3 proved:**
- You served a *real trained model* through **real Triton** and got answers over HTTP.
- Dynamic batching is a **latency↔throughput dial**: it deliberately waits up to `max_queue_delay_microseconds` to
  fuse requests. At concurrency 1 it can't help (nothing to fuse); as concurrency rises the throughput gap widens.
- You verified the **active execution provider** instead of assuming GPU.

**Self-check:** (1) Why does batching barely help at concurrency=1? (2) If your SLO were 'p99 < 10 ms', which way do
you turn `max_queue_delay`? (3) This CNN is compute-bound — why does that make batching effective, and why is LLM
*decode* different? (→ leads into Dynamo, Deliverable 2.)

**Next:** paste the results table into your README, commit (`File → Save a copy in GitHub`), and the resume line is true.
